# 🎙️ InterviewCoach — Analysis Pipeline Demo

This notebook demonstrates the **NLP analysis pipeline** used by InterviewCoach.  
All processing is done **locally** using Python's standard library — no external APIs.

---
### What this notebook covers:
1. Filler word detection
2. WPM (words per minute) calculation
3. Pause detection simulation
4. Keyword extraction
5. Transcript highlighting
6. Full report generation


In [ ]:
import re
from collections import Counter
from IPython.display import HTML, display
import json


## 1. Configuration

In [ ]:
# ── Filler Words ──────────────────────────────────────────────────────────────
FILLERS_EN = [
    "um", "uh", "like", "you know", "basically", "literally", "right",
    "actually", "kind of", "sort of", "honestly", "okay", "i mean", "anyway",
    "you see", "well", "so yeah", "and stuff", "or whatever", "you know what i mean"
]

FILLERS_ID = [
    "ee", "eee", "anu", "gitu", "kan", "tuh", "ya", "jadi", "soalnya", "terus",
    "nah", "intinya", "maksudnya", "gimana", "kayak", "emang", "sebenarnya",
    "begitu", "yah", "loh", "deh", "nih", "sih"
]

# ── Stop Words ────────────────────────────────────────────────────────────────
STOP_WORDS_EN = set([
    "i", "me", "my", "we", "our", "you", "your", "he", "she", "it", "they",
    "them", "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "shall", "can", "to", "of", "in", "on", "at",
    "for", "with", "about", "by", "from", "up", "out", "as", "and", "or",
    "but", "if", "then", "than", "so", "yet", "both", "either", "not", "no",
    "that", "this", "these", "those", "what", "which", "who", "when", "where",
    "how", "why", "there", "here", "just", "also", "very", "really", "more",
    "most", "some", "any", "all"
])

STOP_WORDS_ID = set([
    "saya", "aku", "kita", "kami", "kamu", "anda", "dia", "mereka", "nya",
    "yang", "dan", "atau", "tapi", "juga", "ke", "di", "dari", "untuk",
    "dengan", "pada", "ini", "itu", "ada", "tidak", "bisa", "akan", "sudah",
    "belum", "jika", "kalau", "karena", "oleh", "lebih", "banyak", "sangat",
    "saat", "ketika", "selalu", "sering", "pernah", "dalam", "lagi", "harus",
    "perlu", "bagi", "hal", "cara", "agar", "maka", "serta", "maupun",
    "bahwa", "tentang"
])

print('✅ Configuration loaded')
print(f'  EN fillers: {len(FILLERS_EN)} words')
print(f'  ID fillers: {len(FILLERS_ID)} words')


## 2. Sample Transcript
Replace this with your own transcript to test the pipeline.

In [ ]:
# ── Sample English transcript ─────────────────────────────────────────────────
SAMPLE_TRANSCRIPT_EN = """
So, um, tell me about yourself. Well, I am basically a software engineer with,
you know, about five years of experience. I've worked at, like, three different
companies and, uh, I really enjoy, sort of, building things that help people.
I'm actually quite passionate about, you know, user experience design. I mean,
honestly, I think the best software is software that, um, people actually want
to use. Right? So yeah, that's kind of my background in a nutshell.
""".strip()

# ── Sample Indonesian transcript ──────────────────────────────────────────────
SAMPLE_TRANSCRIPT_ID = """
Jadi, ee, saya adalah seorang pengembang perangkat lunak yang sudah bekerja
selama, ya, sekitar lima tahun. Saya pernah bekerja di, anu, tiga perusahaan
yang berbeda. Intinya, saya sangat suka, gitu, membangun aplikasi yang berguna
bagi pengguna. Maksudnya, saya percaya bahwa teknologi, kan, harus bisa
memudahkan kehidupan. Nah, itu sih gambaran singkat tentang saya.
""".strip()

print('✅ Sample transcripts ready')
print('\n--- English Sample ---')
print(SAMPLE_TRANSCRIPT_EN[:200], '...')


## 3. Analysis Functions

In [ ]:
def count_fillers(text: str, lang: str = 'en') -> dict:
    """Count each filler word occurrence in the transcript."""
    fillers = FILLERS_EN if lang == 'en' else FILLERS_ID
    lower = text.lower()
    result = {}
    # Sort by length descending to match multi-word fillers first
    for f in sorted(fillers, key=len, reverse=True):
        pattern = re.compile(r'\b' + re.escape(f) + r'\b', re.IGNORECASE)
        matches = pattern.findall(lower)
        if matches:
            result[f] = len(matches)
    return result


def calculate_wpm(text: str, duration_seconds: float) -> float:
    """Calculate words per minute."""
    if duration_seconds <= 0:
        return 0.0
    word_count = len(text.split())
    return round((word_count / duration_seconds) * 60, 1)


def extract_keywords(text: str, lang: str = 'en', top_n: int = 12) -> list:
    """Extract most frequent meaningful words, excluding stop words and fillers."""
    stop_words = STOP_WORDS_EN if lang == 'en' else STOP_WORDS_ID
    fillers = set(FILLERS_EN if lang == 'en' else FILLERS_ID)
    
    words = re.findall(r'\b[a-zA-ZÀ-ÿ]{4,}\b', text.lower())
    filtered = [w for w in words if w not in stop_words and w not in fillers]
    freq = Counter(filtered)
    return freq.most_common(top_n)


def highlight_fillers_html(text: str, lang: str = 'en') -> str:
    """Return HTML string with filler words wrapped in highlight spans."""
    fillers = FILLERS_EN if lang == 'en' else FILLERS_ID
    result = text
    style = 'background:#FFF3E0;color:#E65100;border-radius:4px;padding:1px 5px;font-weight:bold;border-bottom:2px solid #FFCC80;'
    for f in sorted(fillers, key=len, reverse=True):
        pattern = re.compile(r'\b(' + re.escape(f) + r')\b', re.IGNORECASE)
        result = pattern.sub(f'<span style="{style}">\\1</span>', result)
    return result


def simulate_pauses(events: list, threshold: float = 1.5) -> list:
    """
    Detect pauses from a list of recognition event timestamps.
    
    Parameters:
        events: list of timestamps (seconds from session start) when speech results arrived
        threshold: minimum gap in seconds to count as a pause
    Returns:
        list of {time, duration} dicts
    """
    pauses = []
    for i in range(1, len(events)):
        gap = events[i] - events[i-1]
        if gap >= threshold:
            pauses.append({'time': round(events[i-1], 1), 'duration': round(gap, 1)})
    return pauses


print('✅ Analysis functions defined')


## 4. Run Analysis — English

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
transcript = SAMPLE_TRANSCRIPT_EN
lang = 'en'
duration_seconds = 52.0  # Simulated session duration

# Simulate recognition event timestamps (seconds from start)
simulated_events = [2.1, 4.5, 7.2, 8.9, 11.3, 14.0, 18.5, 19.1, 22.4, 24.8,
                    27.3, 28.1, 30.9, 34.5, 37.2, 38.8, 42.1, 45.3, 48.7, 51.2]

# ── Run ───────────────────────────────────────────────────────────────────────
words = transcript.split()
word_count = len(words)
wpm = calculate_wpm(transcript, duration_seconds)
fillers_found = count_fillers(transcript, lang)
total_fillers = sum(fillers_found.values())
filler_pct = round((total_fillers / word_count) * 100, 1) if word_count > 0 else 0
keywords = extract_keywords(transcript, lang)
pauses = simulate_pauses(simulated_events, threshold=1.5)

# ── Report ────────────────────────────────────────────────────────────────────
print('═' * 55)
print('  📊 INTERVIEWCOACH SESSION REPORT')
print('═' * 55)
mins = int(duration_seconds // 60)
secs = int(duration_seconds % 60)
print(f'  ⏱️  Duration:      {mins}:{secs:02d}')
print(f'  💬 Word Count:    {word_count}')
print(f'  📈 WPM:           {wpm}')
print(f'  🔴 Filler Words:  {total_fillers} ({filler_pct}% of words)')
print(f'  ⏸️  Pauses:        {len(pauses)} detected')
print(f'  🔑 Keywords:      {len(keywords)} found')
print()

if fillers_found:
    print('  🔴 Filler Breakdown:')
    for word, count in sorted(fillers_found.items(), key=lambda x: -x[1]):
        bar = '█' * count
        print(f'     {word:<20} {bar} ({count}x)')
    print()

if keywords:
    print('  🔑 Top Keywords:')
    for word, count in keywords:
        print(f'     {word:<20} {count}x')
    print()

if pauses:
    print('  ⏸️  Pauses:')
    for p in pauses:
        print(f'     At {p["time"]}s → {p["duration"]}s pause')

print('═' * 55)


## 5. Highlighted Transcript (HTML rendering)

In [ ]:
highlighted = highlight_fillers_html(transcript, lang)

html_output = f"""
<div style="
  font-family: 'Segoe UI', Arial, sans-serif;
  padding: 20px 24px;
  background: linear-gradient(135deg, #E3F2FD, #ffffff);
  border-radius: 12px;
  border: 1.5px solid #90CAF9;
  font-size: 15px;
  line-height: 1.9;
  color: #0D47A1;
  max-width: 800px;
">
  <div style="font-weight:700; font-size:13px; letter-spacing:1px; text-transform:uppercase;
              color:#1565C0; margin-bottom:12px; display:flex; align-items:center; gap:8px;">
    📄 Final Transcript
    <span style="margin-left:auto; font-size:11px; background:#FFF3E0; border:1px solid #FFCC80;
                border-radius:4px; padding:2px 8px; color:#E65100; font-weight:600;
                text-transform:none; letter-spacing:0;">filler = highlighted</span>
  </div>
  {highlighted}
</div>
"""

display(HTML(html_output))


## 6. Run Analysis — Bahasa Indonesia

In [ ]:
transcript_id = SAMPLE_TRANSCRIPT_ID
lang_id = 'id'
duration_id = 38.0

words_id = transcript_id.split()
wc_id = len(words_id)
wpm_id = calculate_wpm(transcript_id, duration_id)
fillers_id = count_fillers(transcript_id, lang_id)
total_fillers_id = sum(fillers_id.values())
filler_pct_id = round((total_fillers_id / wc_id) * 100, 1) if wc_id > 0 else 0
keywords_id = extract_keywords(transcript_id, lang_id)

print('═' * 55)
print('  📊 LAPORAN SESI INTERVIEWCOACH (BAHASA INDONESIA)')
print('═' * 55)
mins_id = int(duration_id // 60)
secs_id = int(duration_id % 60)
print(f'  ⏱️  Durasi:         {mins_id}:{secs_id:02d}')
print(f'  💬 Jumlah Kata:    {wc_id}')
print(f'  📈 KPM:            {wpm_id}')
print(f'  🔴 Kata Filler:    {total_fillers_id} ({filler_pct_id}% dari total kata)')
print(f'  🔑 Kata Kunci:     {len(keywords_id)} ditemukan')
print()

if fillers_id:
    print('  🔴 Rincian Filler:')
    for word, count in sorted(fillers_id.items(), key=lambda x: -x[1]):
        bar = '█' * count
        print(f'     {word:<20} {bar} ({count}x)')
    print()

print('  📄 Transkrip dengan Sorotan:')
highlighted_id = highlight_fillers_html(transcript_id, lang_id)
html_id = f"""
<div style="font-family:'Segoe UI',Arial,sans-serif;padding:16px 20px;
  background:linear-gradient(135deg,#E3F2FD,#fff);border-radius:12px;
  border:1.5px solid #90CAF9;font-size:14px;line-height:1.9;color:#0D47A1;max-width:720px;">
  {highlighted_id}
</div>"""
display(HTML(html_id))


## 7. Export Report as JSON

In [ ]:
report = {
    "language": "en",
    "duration_seconds": duration_seconds,
    "word_count": word_count,
    "wpm": wpm,
    "filler_words": fillers_found,
    "total_fillers": total_fillers,
    "filler_percentage": filler_pct,
    "pauses": pauses,
    "keywords": [{'word': w, 'count': c} for w, c in keywords],
    "transcript": transcript
}

print(json.dumps(report, indent=2))


---
## 🚀 How to Run InterviewCoach Web App

```bash
# Install dependencies
pip install streamlit

# Run the app locally
streamlit run app.py

# Then open: http://localhost:8501
```

### 🌐 Deploy to Streamlit Community Cloud
1. Push this project to GitHub
2. Go to [share.streamlit.io](https://share.streamlit.io)
3. Click **New app** → connect your repo → set `app.py` as the entry point
4. Click **Deploy** ✅
